# Roster Generator

`roster_generator.ipynb` is an ongoing project to create fictional People data employee for an organization roster that can be used to produce the following results:
- Generate employee roster with fields such as: employee_id, hire_date, termination_date, organization, gender, ethnicity, and job_level.
- The generated data is randomly generated using a few certain controls, which are highlighted by the files imported into this notebook: 
	- `period_ratio_map_v1.py` - developer-controlled periods indicating:
		- new hire count, with organization, gender, ethnicity distribution for new hires
		- turnover probability for each period (only applies to employees hired in previous months)
	- `job_level.py` - randomly assigns job levels for new hires
- Notebooks and other works may refer to this fictional data to produce reports, dashboards, hypothesis tests, forecasts, etc. Non-Python work can refer to the CSV file generated by this notebook.


!todo
- add state, country, is_remote

In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
import random
from period_ratio_map_v1 import period_ratio
from job_level import job_level_ratios
from demographics_map import ethnicity_map, non_binary_proportion

# Set a random seed for reproducibility
np.random.seed(42)
random.seed(42)

## Preparation and load

In [2]:
def generate_random_date(start_date, end_date):
    """
    Generate a random business date between start_date and end_date.
    """
    business_days = pd.bdate_range(start=start_date, end=end_date)
    return random.choice(business_days)

def get_end_of_month(date_str):
    """
    Get the end of the month date for a given date string.
    """
    date = pd.to_datetime(date_str)
    return date + pd.offsets.MonthEnd(0)


### Generate `df_periods_and_events`

In [3]:
data_periods = []
for period_start_date, ratios in period_ratio.items():
    period_end_date = get_end_of_month(period_start_date)
    
    row = {
        'period_start_date': pd.to_datetime(period_start_date),
        'period_end_date': period_end_date,
        'new_hire': ratios['new_hire'],
        'attrition': ratios['attrition'],
    }
    
    for org, ratio in ratios['organizations'].items():
        row[f'org-{org}'] = ratio
        
    for gender, ratio in ratios['gender'].items():
        row[f'gender-{gender}'] = ratio
        
    for ethnicity, ratio in ratios['ethnicity'].items():
        row[f'ethnicity-{ethnicity}'] = ratio
        
    data_periods.append(row)

df_periods_and_events = pd.DataFrame(data_periods)


In [4]:
print("df_periods_and_events:")
df_periods_and_events


df_periods_and_events:


,period_start_date,period_end_date,new_hire,attrition,org-Administrative,org-Production,org-Sales,gender-Male,gender-Female,ethnicity-White,ethnicity-Asian,ethnicity-Black,ethnicity-Hispanic
0,2014-01-01,2014-01-31,10,0.00,1.0,0.0,0.0,0.7,0.3,0.6,0.2,0.1,0.1
1,2014-02-01,2014-02-28,6,0.00,1.0,0.0,0.0,0.7,0.3,0.6,0.2,0.1,0.1
2,2014-03-01,2014-03-31,0,0.00,1.0,0.0,0.0,0.7,0.3,0.6,0.2,0.1,0.1
3,2014-04-01,2014-04-30,1,0.00,0.7,0.3,0.0,0.7,0.3,0.6,0.2,0.1,0.1
4,2014-05-01,2014-05-31,9,0.00,0.7,0.3,0.0,0.7,0.3,0.6,0.2,0.1,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,2024-07-01,2024-07-31,45,0.00,0.3,0.3,0.4,0.4,0.6,0.2,0.4,0.2,0.2
127,2024-08-01,2024-08-31,60,0.00,0.3,0.3,0.4,0.4,0.6,0.2,0.4,0.2,0.2
128,2024-09-01,2024-09-30,132,0.01,0.3,0.3,0.4,0.4,0.6,0.2,0.4,0.2,0.2
129,2024-10-01,2024-10-31,69,0.04,0.3,0.3,0.4,0.4,0.6,0.2,0.4,0.2,0.2


### Generate `df_employees`

In [5]:
df_test = pd.DataFrame(
    [{
        'employee_id': 'e000001',
        'hire_date': pd.to_datetime('2014-01-02'),
        'termination_date': pd.NaT,
        'org_l01': '',
        'gender': 'Male',
        'ethnicity': 'White',
        'job_level': 'M11',
    }],
    columns=['employee_id', 'hire_date', 'termination_date', 'org_l01', 'gender', 'ethnicity', 'job_level',],
    # dtype=[]
)
df_test.loc[len(df_test)] = [
    'e000002',
    pd.to_datetime('2014-01-03'),
    np.nan,
    'Administrative',
    'Female',
    'White',
    'IC4'
]
df_test.loc[df_test.employee_id == 'e000001'] = [
    'e000001',
    pd.to_datetime('2014-01-02'),
    np.nan,
    'Administrative',
    'Male',
    'White',
    'M11'
]
display(df_test)
df_test.loc[df_test.employee_id == 'e000001'] = [
    'e000001',
    pd.to_datetime('2014-01-02'),
    np.nan,
    '',
    'Male',
    'White',
    'M11'
]
df_test

,employee_id,hire_date,termination_date,org_l01,gender,ethnicity,job_level
0,e000001,2014-01-02,NaN,Administrative,Male,White,M11
1,e000002,2014-01-03,NaN,Administrative,Female,White,IC4


,employee_id,hire_date,termination_date,org_l01,gender,ethnicity,job_level
0,e000001,2014-01-02,NaN,,Male,White,M11
1,e000002,2014-01-03,NaN,Administrative,Female,White,IC4


In [6]:
df_test = pd.DataFrame(
    [{
        'a': 1,
        'b': 2
    }],
    columns=['a', 'b']
)
df_test

,a,b
0,1,2


In [7]:
# -----------------------------------------------------------------------------
# Generate df_employees
# -----------------------------------------------------------------------------

employee_records = []
employee_id_counter = 2
# df_employees = pd.DataFrame(columns=['employee_id', 'hire_date', 'termination_date', 'org_l01', 'gender', 'ethnicity', 'job_level'])
df_employees = pd.DataFrame(
    [{
        'employee_id': 'e000001',
        'hire_date': pd.to_datetime('2014-01-02'),
        'termination_date': pd.NaT,
        'org_l01': '',
        'gender': 'Male',
        'ethnicity': 'White',
        'job_level': 'M11',
    }],
    columns=['employee_id', 'hire_date', 'termination_date', 'org_l01', 'gender', 'ethnicity', 'job_level',]
)

for index, row in df_periods_and_events.iterrows():
    period_start_date = row['period_start_date']
    period_end_date = row['period_end_date']
    num_new_hires = int(row['new_hire'])
    attrition_rate = row['attrition']

    # Prepare ratio lists for random choices
    org_ratios = {k.replace('org-', ''): v for k, v in row.items() if k.startswith('org-')}
    gender_ratios = {k.replace('gender-', ''): v for k, v in row.items() if k.startswith('gender-')}
    ethnicity_ratios = {k.replace('ethnicity-', ''): v for k, v in row.items() if k.startswith('ethnicity-')}
    
    org_choices = list(org_ratios.keys())
    org_weights = list(org_ratios.values())
    
    gender_choices = list(gender_ratios.keys())
    gender_weights = list(gender_ratios.values())
    
    ethnicity_choices = list(ethnicity_ratios.keys())
    ethnicity_weights = list(ethnicity_ratios.values())
    
    job_level_choices = list(job_level_ratios.keys())
    job_level_weights = list(job_level_ratios.values())

    # Process attrition for existing employees (those hired before the current period)
    if not df_employees.empty:
        # Get employees who have not been terminated and were hired before the current period
        active_employees = df_employees[df_employees['termination_date'].isna() & (df_employees['hire_date'] < period_start_date)]
        
        for emp_index, emp_row in active_employees.iterrows():
            if random.random() < attrition_rate:
                # Assign a termination date between the current period's start and end dates
                term_date = generate_random_date(period_start_date, period_end_date)
                df_employees.loc[emp_index, 'termination_date'] = term_date
                
    # Generate new hires for the current period
    new_hires = []
    for _ in range(num_new_hires):
        employee_id = f'e{employee_id_counter:06d}'
        hire_date = generate_random_date(period_start_date, period_end_date)
        
        # New hires in the current period do not have a termination date yet
        termination_date = np.nan
        
        org = random.choices(org_choices, weights=org_weights, k=1)[0]
        gender = random.choices(gender_choices, weights=gender_weights, k=1)[0]
        ethnicity = random.choices(ethnicity_choices, weights=ethnicity_weights, k=1)[0]
        # job_level_val = random.choice(job_level)
        job_level_val = random.choices(job_level_choices, weights=job_level_weights, k=1)[0]
        
        new_hires.append({
            'employee_id': employee_id,
            'hire_date': hire_date,
            'termination_date': termination_date,
            'org_l01': org,
            'gender': gender,
            'ethnicity': ethnicity,
            'job_level': job_level_val,
        })
        
        employee_id_counter += 1
    
    # Append the new hires to the main employee DataFrame
    if new_hires:
        df_new_hires = pd.DataFrame(new_hires)
        df_employees = pd.concat([df_employees, df_new_hires], ignore_index=True)

# Final formatting
df_employees['hire_date'] = pd.to_datetime(df_employees['hire_date'])
df_employees['termination_date'] = pd.to_datetime(df_employees['termination_date'])

# Resetting CEO attributes
df_employees.loc[df_employees.employee_id == 'e000001'] = [
    'e000001',
    pd.to_datetime('2014-01-02'),
    np.nan,
    '',
    'Male',
    'White',
    'M11'
]

df_employees['ethnicity_is_underrepresented_minority'] = df_employees['ethnicity'].isin(ethnicity_map['URM'])
df_employees.reset_index(drop=True, inplace=True)
df_employees.sort_values(by=['employee_id'], inplace=True)

#### Randomly relabeling `gender` field using a proportion

In [8]:
# identify size
size = int(max(round(len(df_employees) * non_binary_proportion), 1))
# randomly choose indices for replacement
indices_to_relabel = np.random.choice(df_employees.index, size=size, replace=False)
# updating gender using randomly generated proportion
df_employees.loc[indices_to_relabel, 'gender'] = 'Non-binary'
# df_employees

<summary> Old `df_employees`</summary>
<details>
<code>
# -----------------------------------------------------------------------------
# Generate df_employees
# -----------------------------------------------------------------------------

employee_records = []
employee_id_counter = 2

for index, row in df_periods_and_events.iterrows():
    period_start_date = row['period_start_date']
    period_end_date = row['period_end_date']
    num_new_hires = int(row['new_hire'])
    attrition_rate = row['attrition']

    '''Prepare ratio lists for random choices'''
    org_ratios = {k.replace('org-', ''): v for k, v in row.items() if k.startswith('org-')}
    gender_ratios = {k.replace('gender-', ''): v for k, v in row.items() if k.startswith('gender-')}
    ethnicity_ratios = {k.replace('ethnicity-', ''): v for k, v in row.items() if k.startswith('ethnicity-')}
    
    org_choices = list(org_ratios.keys())
    org_weights = list(org_ratios.values())
    
    gender_choices = list(gender_ratios.keys())
    gender_weights = list(gender_ratios.values())
    
    ethnicity_choices = list(ethnicity_ratios.keys())
    ethnicity_weights = list(ethnicity_ratios.values())

    for _ in range(num_new_hires):
        employee_id = f'e{employee_id_counter:06d}'
        hire_date = generate_random_date(period_start_date, period_end_date)
        
        termination_date = np.nan
        if (random.random() < attrition_rate) & (hire_date >= period_end_date):
            termination_date = generate_random_date(hire_date, period_end_date)
        
        org = random.choices(org_choices, weights=org_weights, k=1)[0]
        gender = random.choices(gender_choices, weights=gender_weights, k=1)[0]
        ethnicity = random.choices(ethnicity_choices, weights=ethnicity_weights, k=1)[0]
        job_level_val = random.choice(job_level)
        
        employee_records.append({
            'employee_id': employee_id,
            'hire_date': hire_date,
            'termination_date': termination_date,
            'org_l01': org,
            'gender': gender,
            'ethnicity': ethnicity,
            'job_level': job_level_val,
        })
        
        employee_id_counter += 1

employee_records = [{
    'employee_id': 'e000001',
    'hire_date': pd.to_datetime('2014-01-02'),
    'termination_date': np.nan,
    'org_l01': '',
    'gender': 'Male',
    'ethnicity': 'White',
    'job_level': 'M11'
}]
df_employees = pd.DataFrame(employee_records)

'''Convert date fields to datetime objects'''
df_employees['hire_date'] = pd.to_datetime(df_employees['hire_date'])
df_employees['termination_date'] = pd.to_datetime(df_employees['termination_date'])
</code>
</details>

### Validations

In [9]:
display(df_employees.head())
display(df_employees.tail())

,employee_id,hire_date,termination_date,org_l01,gender,ethnicity,job_level,ethnicity_is_underrepresented_minority
0,e000001,2014-01-02,NaT,,Male,White,M11,False
1,e000002,2014-01-29,2019-09-19,Administrative,Female,White,S2,False
2,e000003,2014-01-06,NaT,Administrative,Female,White,IC2,False
3,e000004,2014-01-01,2022-12-26,Administrative,Male,Asian,IC4,False
4,e000005,2014-01-31,2018-12-05,Administrative,Male,White,IC4,False


,employee_id,hire_date,termination_date,org_l01,gender,ethnicity,job_level,ethnicity_is_underrepresented_minority
5720,e005721,2024-11-28,NaT,Sales,Female,White,IC1,False
5721,e005722,2024-11-25,NaT,Administrative,Male,Asian,S2,False
5722,e005723,2024-11-21,NaT,Production,Male,Asian,IC2,False
5723,e005724,2024-11-12,NaT,Sales,Male,White,IC3,False
5724,e005725,2024-11-25,NaT,Sales,Male,Hispanic,IC3,True


#### Inspect monthly headcount on and since 2021-12-31

In [10]:
df_employees[
    (df_employees.hire_date <= '2021-12-31')
    & (df_employees.termination_date >= '2021-12-31')
].employee_id.nunique()

1023

#### Inspecting monthly results

In [11]:
monthly_results = df_periods_and_events[(df_periods_and_events.period_end_date >= '2021-12-31')][['period_start_date', 'period_end_date']].reset_index(drop=True).copy()
monthly_results['n_active'] = 0
monthly_results['n_new_hires'] = 0
monthly_results['n_terminations'] = 0
for idx, row in monthly_results.iterrows():
    monthly_results.loc[idx, 'n_active'] = df_employees[
        (df_employees.hire_date <= row.iloc[1]) 
        & (
            (df_employees.termination_date >= row.iloc[1]) 
            | (df_employees.termination_date.isnull())
        )
    ].employee_id.nunique()
    
    monthly_results.loc[idx, 'n_new_hires'] = df_employees[
        (df_employees.hire_date >= row.iloc[0]) 
        & (df_employees.hire_date <= row.iloc[1])
    ].employee_id.nunique()
    
    monthly_results.loc[idx, 'n_terminations'] = df_employees[
        (df_employees.termination_date >= row.iloc[0]) 
        & (df_employees.termination_date <= row.iloc[1])
    ].employee_id.nunique()

monthly_results

,period_start_date,period_end_date,n_active,n_new_hires,n_terminations
0,2021-12-01,2021-12-31,1457,7,15
1,2022-01-01,2022-01-31,1411,5,52
2,2022-02-01,2022-02-28,1517,196,91
3,2022-03-01,2022-03-31,1581,66,0
4,2022-04-01,2022-04-30,1599,128,110
5,2022-05-01,2022-05-31,1604,45,43
6,2022-06-01,2022-06-30,1602,42,47
7,2022-07-01,2022-07-31,1559,10,47
8,2022-08-01,2022-08-31,1564,20,15
9,2022-09-01,2022-09-30,1748,184,0


#### Inspect specific employee_id's

In [12]:
df_employees[df_employees.employee_id.isin(['e000002'])]

,employee_id,hire_date,termination_date,org_l01,gender,ethnicity,job_level,ethnicity_is_underrepresented_minority
1,e000002,2014-01-29,2019-09-19,Administrative,Female,White,S2,False


### Save to CSV

In [13]:
df_employees.to_csv('employee_data.csv', index=False)